# 01 · Can PointLLM recognise our scans?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Md-Mosabbir/localpoint3d/blob/main/notebooks/01_pointllm_recognition_test.ipynb)

**Goal:** feed our own iPhone scans (Scaniverse) to **PointLLM-7B** and see what it says. First with an everyday object (a chair) to check the pipeline works, then with local objects (rickshaw, thelagari, …).

**What happens in this notebook**
1. Set up: GPU check, get the code, build PointLLM's Python environment
2. Load our scans and look at what the model will actually see
3. Start the model and ask questions, one at a time or in a batch → CSV

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then run the cells top to bottom (Shift+Enter). The first run takes ~15–20 min (installs + a ~14 GB model download).

---
# Part 1 · Setup

## Check the GPU

`nvidia-smi` lists the graphics card. We need roughly 14 GB of GPU memory for the 7B model in float16; a T4 has 15 GB. `free -h` shows normal RAM.

> 🧠 Lines starting with `!` are not Python. They are run as shell (terminal) commands.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -h | head -2

## Get our project code

Colab only opened this *notebook*. Our helper code lives in the repo's `src/` folder, so we clone the repo into the Colab machine (or update it if it's already there).

> 🧠 **Python note:** `sys.path` is the list of folders Python searches when you write `import something`. Adding `src/` to it is what lets us `import pc_io` below.

In [ ]:
import os, sys

REPO_DIR = "/content/localpoint3d"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 https://github.com/Md-Mosabbir/localpoint3d.git {REPO_DIR}

sys.path.insert(0, f"{REPO_DIR}/src")
print("Code is in", REPO_DIR)

## Get PointLLM

This is the official PointLLM code. We use HTTPS because the `git@…` SSH link in their README doesn't work on Colab.

In [ ]:
POINTLLM_DIR = "/content/PointLLM"
if not os.path.isdir(POINTLLM_DIR):
    !git clone --depth 1 https://github.com/InternRobotics/PointLLM.git {POINTLLM_DIR}
else:
    print("PointLLM already cloned")

## Build a separate Python 3.10 environment

PointLLM was written for **Python 3.10 + PyTorch 2.0.1 + an old Transformers**. Colab's own Python is newer and those versions won't install into it. So we make a separate, isolated Python 3.10 environment (a *virtual environment*, "venv") at `/content/venv` using `uv`, a fast installer.

The notebook itself keeps running on Colab's Python. Later we start the model **inside the venv** as a separate background process.

> 🧠 `%%bash` at the top turns the whole cell into a shell script. `set -e` = stop at the first error.

In [ ]:
%%bash
set -e
pip -q install uv
uv venv --python 3.10 /content/venv

## Install PointLLM's dependencies into that environment (~5 min)

- `torch==2.0.1` matches what PointLLM was tested with
- `-e .` installs the PointLLM repo itself plus everything in its requirements
- `plyfile`, `trimesh` are for reading our scan files (used by `src/pc_io.py`)
- `numpy<2` everywhere because the old libraries break with NumPy 2

In [ ]:
%%bash
set -e
cd /content/PointLLM
PY=/content/venv/bin/python
uv pip install --python $PY "torch==2.0.1" "numpy<2"
uv pip install --python $PY -e . "numpy<2"
uv pip install --python $PY accelerate plyfile trimesh matplotlib "numpy<2"

## Smoke test

Quick check that everything imports **before** we spend 10 minutes downloading model weights. You should see `imports OK`. Warnings about deprecated functions are fine.

In [ ]:
%%bash
cd /content/PointLLM
/content/venv/bin/python -c "
import torch, transformers, pointllm, plyfile, trimesh
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)
print('imports OK')"

---
# Part 2 · Our scans

## Install the file readers in Colab's own Python

The venv has them already. The notebook's Python needs them too, so we can preview scans here.

In [ ]:
!pip -q install plyfile trimesh

## Get the scans

Two options. Pick one by setting `USE_DRIVE` in the cell below.

**A. Google Drive (recommended for the team).** Keep one shared folder `localpoint3d/scans` on Drive. Each of us right-clicks the shared folder → *Organize* → *Add shortcut* → *My Drive*, so the path is the same for all three of us. Files stay there between Colab sessions.

**B. Upload from your computer.** Quick, but the files disappear when the Colab session ends.

Put each object in its own folder, e.g. `scans/Chair/Chair_1.ply`. Subfolders are searched automatically. See `data/README.md` in the repo.

Exported **OBJ**? Keep its `.mtl` and texture image (`.jpg`/`.png`) in the same folder, otherwise the colour is lost.

In [ ]:
import glob

USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/localpoint3d"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SCANS_DIR = f"{DRIVE_ROOT}/scans"
    RESULTS_DIR = f"{DRIVE_ROOT}/results"
else:
    from google.colab import files
    SCANS_DIR, RESULTS_DIR = "/content/scans", "/content/results"
    os.makedirs(SCANS_DIR, exist_ok=True)
    for name, data in files.upload().items():  # opens a file picker
        with open(os.path.join(SCANS_DIR, name), "wb") as fh:
            fh.write(data)

os.makedirs(RESULTS_DIR, exist_ok=True)
EXTS = (".ply", ".obj", ".glb", ".gltf", ".off", ".stl")
# ** + recursive=True = also look inside subfolders (Chair/, Cup/, ...)
SCANS = sorted(p for p in glob.glob(f"{SCANS_DIR}/**/*", recursive=True) if p.lower().endswith(EXTS))
print(f"{len(SCANS)} scan(s) in {SCANS_DIR}")
for p in SCANS:
    print("  ", os.path.relpath(p, SCANS_DIR))

## What is in each file?

`describe()` (from `src/pc_io.py`) tells us the file type, how many points, whether it has colour, and its size.

- **Scaniverse** exports in metres, so a rickshaw should be about 2.5–3 m long.
- Scaniverse **Mesh** exports (`.ply` / `.obj`) are triangle meshes. We sample 100k points on the surface to turn them into a point cloud.

In [ ]:
from pc_io import describe, parse_scan_name

for p in SCANS:
    try:
        print(describe(p))
    except Exception as e:
        print(os.path.basename(p), "-> could not read:", e)

## See what the model will see

PointLLM always gets exactly **8192 points**, centred and squashed into a unit sphere, with RGB colour. `preprocess()` does that:
1. load the file (sample the surface if it's a mesh)
2. pick 8192 well-spread points (*farthest point sampling*)
3. optionally rotate, then centre and scale

If an object looks lying-down or upside-down in the X-Z view, try another `ROTATION`. iPhone scans are **Y-up**, and many 3D datasets are **Z-up**, so `"+90 deg about X"` is often the fix.

In [ ]:
from IPython.display import display
from pointllm_helpers import preprocess, preview, ROTATIONS, PAPER_PROMPTS

ROTATION = "None"   # options: list(ROTATIONS)

for p in SCANS:
    pts, has_color = preprocess(p, rotation=ROTATION)
    display(preview(pts, title=os.path.basename(p) + ("" if has_color else "  (no colour)")))

---
# Part 3 · Ask PointLLM

## Start the model in the background

This starts `src/pointllm_worker.py` with the venv's Python 3.10. The worker loads the model **once** (the first time it also downloads ~14 GB) and then waits for questions. The helper `ask_pointllm()` sends it a question and waits for the answer.

> 🧠 `subprocess.Popen` starts another program and keeps a "pipe" to it. We write questions into its `stdin` and read answers from its `stdout`, one JSON line each way.

In [ ]:
import json, subprocess

def start_worker():
    env = {**os.environ, "POINTLLM_DIR": POINTLLM_DIR}
    w = subprocess.Popen(["/content/venv/bin/python", "-u", f"{REPO_DIR}/src/pointllm_worker.py"],
                         cwd=POINTLLM_DIR, env=env, text=True, bufsize=1,
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print("Loading the model (first time: downloads ~14 GB, be patient)...")
    for line in w.stdout:
        if "@@READY@@" in line:
            print("✅ Model ready.")
            return w
        if line.strip():
            print("  ", line.rstrip()[:200])
    raise RuntimeError("The worker stopped while loading. Read the log above.")


def ask_pointllm(path, prompt, rotation="None", sample=False):
    worker.stdin.write(json.dumps({"path": path, "prompt": prompt, "rotation": rotation, "sample": sample}) + "\n")
    worker.stdin.flush()
    for line in worker.stdout:
        if line.startswith("@@RESULT@@"):
            res = json.loads(line[len("@@RESULT@@"):])
            if "error" in res:
                raise RuntimeError(res["error"])
            return res["answer"]
    raise RuntimeError("The worker died. Re-run this cell to restart it.")


if "worker" not in globals() or worker.poll() is not None:  # don't load twice
    worker = start_worker()

## Ask one question

Change `SCAN` and `QUESTION` and re-run this cell as often as you like. The model stays loaded.

The three prompts from the PointLLM paper are in `PAPER_PROMPTS`:
- `"What is this?"`
- `"This is an object of "`: the model completes the sentence
- `"Caption this 3D model in detail."`

In [ ]:
SCAN = SCANS[0]
QUESTION = "What is this?"

print(os.path.basename(SCAN), "|", QUESTION)
print("PointLLM:", ask_pointllm(SCAN, QUESTION, rotation=ROTATION))

## Batch run: every scan × every paper prompt → CSV

This is the cell that produces our **results**. Each row records which object, which capture method, which rotation and prompt, and what PointLLM answered. The CSV goes to `RESULTS_DIR` (on Drive if you used Drive).

`ROTATIONS_TO_TRY` lets us test whether orientation changes the answer.

In [ ]:
import pandas as pd
from datetime import datetime

ROTATIONS_TO_TRY = [ROTATION]          # e.g. ["None", "+90 deg about X"]
PROMPTS_TO_TRY = PAPER_PROMPTS

rows = []
for p in SCANS:
    meta = parse_scan_name(p)
    for rot in ROTATIONS_TO_TRY:
        for q in PROMPTS_TO_TRY:
            ans = ask_pointllm(p, q, rotation=rot)
            rows.append({"file": os.path.basename(p), **meta, "rotation": rot, "prompt": q, "answer": ans})
            print(f"{os.path.basename(p)} | {rot} | {q!r}\n   -> {ans}\n")

results = pd.DataFrame(rows)
out_csv = f"{RESULTS_DIR}/pointllm_{datetime.now():%Y%m%d_%H%M}.csv"
results.to_csv(out_csv, index=False)
print("Saved", out_csv)
results

## (Optional) Free chat about one scan

Type questions. Press **Enter on an empty line** to stop.

In [ ]:
SCAN = SCANS[0]
print("Chatting about", os.path.basename(SCAN))
while True:
    q = input("You > ").strip()
    if not q:
        break
    print("PointLLM:", ask_pointllm(SCAN, q, rotation=ROTATION))

## Stop the model (frees the GPU)

Do this when you're done, or before switching to another model. Colab also frees everything when the runtime disconnects.

In [ ]:
worker.terminate()
print("Worker stopped.")

---
## Troubleshooting

| Problem | Try |
|---|---|
| `CUDA out of memory` | Runtime → Restart session, run again. Make sure no other worker is running. |
| Crash while loading, RAM full | System RAM ran out. Use a High-RAM runtime (Colab Pro) or retry. |
| Answers look random | Look at the preview. Is the object upright and are the background and ground cropped away? Try another `ROTATION`. |
| Worker error mentions an import | PointLLM's API changed. Compare `src/pointllm_helpers.py` with `PointLLM/pointllm/eval/PointLLM_chat.py`. |